# 02 — Model Training

**PRD Phase 2.** Trains the three PRD models, all **unsupervised / normal-only**:

| Model | Role | Track | PRD |
|---|---|---|---|
| Isolation Forest | primary | NSL-KDD tabular | 7.2.1 |
| LOF | baseline | NSL-KDD tabular | 7.2.3 |
| LSTM autoencoder | time-series | synthetic stream | 7.2.2 |

Labels are used **only** for the quick sanity-check at the bottom — never for fitting. Full evaluation is Phase 3 (`03_Evaluation.ipynb`).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
print('root:', PROJECT_ROOT)

## Track A — Isolation Forest (primary) + LOF (baseline)

In [ ]:
from data_loader import load_dataset
from preprocessing import unsupervised_split, save_artifacts
from models.isolation_forest import IsolationForestDetector
from models.lof import LOFDetector

df = load_dataset()
X_train, X_test, y_test, scaler, encoders = unsupervised_split(df)
save_artifacts(scaler, encoders, X_train, X_test, y_test)
print('normal-only train:', X_train.shape, '| test:', X_test.shape,
      '| test anomalies:', int((y_test == 1).sum()))

In [ ]:
iforest = IsolationForestDetector().fit(X_train)
iforest.save()
print('IF threshold:', round(iforest.threshold_, 4))
print('Latency ms/record:', round(iforest.measure_latency(X_test.values), 4), '(target <100ms)')

lof = LOFDetector().fit(X_train)
lof.save()
print('LOF threshold:', round(lof.threshold_, 4))

In [ ]:
# Quick sanity check (NOT the full eval) — recall/precision at default thresholds
from sklearn.metrics import precision_score, recall_score, roc_auc_score
for name, det in [('IsolationForest', iforest), ('LOF', lof)]:
    pred = det.predict(X_test)
    score = det.anomaly_score(X_test)
    print(f"{name:16s} P={precision_score(y_test, pred):.3f} "
          f"R={recall_score(y_test, pred):.3f} "
          f"AUC={roc_auc_score(y_test, score):.3f}")

## Track B — LSTM autoencoder (time-series)
Requires TensorFlow and the synthetic stream features.

In [ ]:
stream_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'stream_features.csv')
if not os.path.exists(stream_path):
    print('Run: python src/synthetic_stream.py && python src/feature_engineering.py')
else:
    try:
        import tensorflow as tf
        from sklearn.preprocessing import StandardScaler
        from feature_engineering import build_features, ENGINEERED_NUMERIC
        from models.lstm_autoencoder import LSTMAutoencoderDetector
        feats = build_features()
        normal = feats[feats['label'] == 0]
        sc = StandardScaler().fit(normal[ENGINEERED_NUMERIC])
        Xn = sc.transform(normal[ENGINEERED_NUMERIC])
        lstm = LSTMAutoencoderDetector(epochs=20).fit(Xn, verbose=1)
        lstm.save()
        print('LSTM recon threshold:', round(lstm.threshold_, 6))
    except ImportError:
        print('TensorFlow not installed: pip install tensorflow')

## Done
Models saved to `models/`:
* `isolation_forest_model.pkl` (+ `if_metadata.json`)
* `lof_model.pkl`
* `lstm_autoencoder.h5` (+ `lstm_metadata.json`)

Proceed to **Phase 3 — Evaluation** for the full metrics, ROC/PR curves, threshold tuning, and the model-comparison table.